In [ ]:
# # ================================================================
# # CELL 1 — Install Library
# # ================================================================
!pip install -q gradio catboost groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 12.1 MB/s eta 0:00:00


In [ ]:
# ================================================================
# CELL 2 — Import Library
# ================================================================
import re
import numpy as np
import pandas as pd
import gradio as gr
from catboost import CatBoostRegressor
from joblib import load
from groq import Groq

print("✅ Library berhasil diimport.")

✅ Library berhasil diimport.


In [ ]:
# # ================================================================
# # CELL 3 — Load Model & Artefak Training
# # ================================================================
# # Load model CatBoost
# cat_model = CatBoostRegressor()
# cat_model.load_model("catboost_model_rev (1).cbm")
# print("✅ Model CatBoost berhasil dimuat.")

# # Load daftar nama fitur (harus sama persis dengan saat training)
# feature_names = load("feature_names_rev (1).pkl")
# print(f"Jumlah fitur model : {len(feature_names)}")

# # Load rare-values mapping (untuk handle nilai langka saat prediksi)
# rare_map = load("rare_values_mapping (2).pkl")
# print("✅ Rare-values mapping berhasil dimuat.")

In [ ]:
# ================================================================
# CELL 3 — Load Model & Artefak Training
# ================================================================
import joblib

# 1. Load model Random Forest Terbaik
rf_model = joblib.load("rf_model_rev3.pkl")
print("✅ Model Random Forest berhasil dimuat.")

# 2. Load daftar nama fitur (harus sama persis dengan saat training)
feature_names = joblib.load("feature_names_rev3.pkl")
print(f"✅ Jumlah fitur model : {len(feature_names)}")

# 3. Load rare-values mapping (untuk menangani kategori langka)
rare_map = joblib.load("rare_values_mapping3.pkl")
print("✅ Rare-values mapping berhasil dimuat.")

✅ Model Random Forest berhasil dimuat.
✅ Jumlah fitur model : 113
✅ Rare-values mapping berhasil dimuat.


In [ ]:
# ================================================================
# CELL 4 — Load Groq Client (AI Insight)
# ================================================================
GROQ_API_KEY = "YOUR_API_KEY"
groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq siap digunakan.")

✅ Groq siap digunakan.


In [ ]:
# ================================================================
# CELL 5 — Load Referensi CSV & Bangun Daftar Pilihan UI
# ================================================================
feature_df = pd.read_csv("Dataset_Raw_Scraping_Updated7.csv")

brands = sorted([
    c.replace("brand_", "")
    for c in feature_names
    if c.startswith("brand_") and c != "brand_Other"
])

locations = sorted([
    c.replace("location_", "")
    for c in feature_names
    if c.startswith("location_") and c != "location_Other"
])

print(f"Brand tersedia     : {len(brands)}")
print(f"Lokasi tersedia    : {len(locations)}")

Brand tersedia     : 13
Lokasi tersedia    : 26


In [ ]:
# ================================================================
# CELL 6 — Fungsi Helper
# ================================================================

# ---- 6.1 Terapkan Rare-Values Mapping ----
# Nilai yang tidak dikenal model → diganti "Other"
def apply_rare_map(value: str, col: str) -> str:
    rare_vals = rare_map.get(col, [])
    return "Other" if value in rare_vals else value

# ---- 6.2 Normalisasi Nama Model ----
# Hapus angka kapasitas mesin di akhir (contoh: "Avanza 1.5" → "Avanza")
def normalize_model_name(name: str) -> str:
    return re.sub(r"\s+\d+\.\d+$", "", name).strip()

# ---- 6.3 Update Dropdown Model saat Brand dipilih ----
def update_models(selected_brand):
    models = sorted(
        feature_df[feature_df["brand"] == selected_brand]["car name"].unique()
    )
    return gr.update(choices=models, value=None)

# ---- 6.4 Update Dropdown Tahun saat Model dipilih ----
def update_years(selected_brand, selected_model):
    years = sorted(
        feature_df[
            (feature_df["brand"] == selected_brand) &
            (feature_df["car name"] == selected_model)
        ]["year"].unique().tolist()
    )
    return gr.update(choices=years, value=None)

# ---- 6.5 Auto-fill Engine CC saat Tahun dipilih ----
def auto_fill_engine_cc(car_model, year):
    if car_model is None or year is None:
        return 1500
    match = re.search(r"(\d+(?:\.\d+)?)\s*$", str(car_model))
    if match:
        cc_val = float(match.group(1))
        return int(cc_val * 1000) if cc_val < 10 else int(cc_val)
    return 1500

print("✅ Fungsi helper berhasil didefinisikan.")

✅ Fungsi helper berhasil didefinisikan.


In [ ]:
# ================================================================
# CELL 7 — Fungsi Persiapan Input (prepare_input)
# ================================================================
def prepare_input(
    brand,
    car_model,
    location,
    mileage,
    transmission,
    car_age,
    engine_cc,
):
    # Terapkan rare-values mapping sebelum encoding
    brand_safe     = apply_rare_map(brand, "brand")
    car_model_safe = apply_rare_map(normalize_model_name(car_model), "car_model")
    location_safe  = apply_rare_map(location, "location")
    transmission_safe = apply_rare_map(transmission, "transmission")

    # Buat dataframe kosong sesuai struktur fitur model
    data = pd.DataFrame(
        np.zeros((1, len(feature_names))),
        columns=feature_names
    )

    # ---- Fitur Numerik ----
    data["mileage (km)"] = mileage
    data["engine_cc"]    = engine_cc
    data["car_age"]      = car_age

    # ---- One-Hot: Brand ----
    brand_col = f"brand_{brand_safe}"
    if brand_col in data.columns:
        data[brand_col] = 1

    # ---- One-Hot: Car Model ----
    model_col = f"car_model_{car_model_safe}"
    if model_col in data.columns:
        data[model_col] = 1

    # ---- One-Hot: Location ----
    loc_col = f"location_{location_safe}"
    if loc_col in data.columns:
        data[loc_col] = 1

    # ---- One-Hot: Transmission ----
    trans_col = f"transmission_{transmission_safe}"
    if trans_col in data.columns:
        data[trans_col] = 1

    return data

print("✅ Fungsi prepare_input berhasil didefinisikan.")

✅ Fungsi prepare_input berhasil didefinisikan.


In [ ]:
def generate_ai_insight(
    brand, car_model, year, location,
    mileage, transmission, engine_cc, predicted_price
):
    prompt = f"""
Anda adalah analis harga mobil bekas profesional di Indonesia.

Data kendaraan yang dianalisis:
- Brand       : {brand}
- Model       : {car_model}
- Tahun       : {year}
- Lokasi      : {location}
- Transmisi   : {transmission}
- Kapasitas   : {engine_cc} cc
- Mileage     : {mileage:,.0f} km

Prediksi harga oleh model Random Forest: {predicted_price}

Berikan analisis singkat dalam Bahasa Indonesia dengan format:

## 🔍 Analisis Harga
(Apakah harga wajar? Kenapa?)

## 📌 Faktor yang Mempengaruhi
(Sebutkan 3–4 faktor utama dari data di atas)

## 💡 Saran
(Saran untuk pembeli atau penjual)
"""
    try:
        resp = groq_client.chat.completions.create(
            model="qwen/qwen3.6-27b",
            messages=[{"role": "user", "content": prompt}],
            reasoning_format="hidden",   # jangan kembalikan token <think>...</think>, cuma jawaban final
            reasoning_effort="none",     # matikan reasoning: tugas ini cukup sederhana, jadi lebih cepat & tidak berisiko kehabisan token
            max_completion_tokens=1024,  # kasih ruang cukup supaya jawaban tidak terpotong
            temperature=0.6,
        )
        content = resp.choices[0].message.content
        if not content or not content.strip():
            return "⚠️ AI Insight kosong — coba klik Prediksi lagi atau perbesar `max_completion_tokens`."
        return content
    except Exception as e:
        return f"⚠️ AI Error: {e}"

In [ ]:
# ================================================================
# CELL 9 — Fungsi Prediksi Utama (predict_car_price)
# ================================================================
def predict_car_price(
    brand, car_model, year,
    location,
    mileage, transmission, engine_cc,
):
    CURRENT_YEAR = 2026
    car_age = CURRENT_YEAR - int(year)

    input_df = prepare_input(
        brand=brand,
        car_model=car_model,
        location=location,
        mileage=mileage,
        transmission=transmission,
        car_age=car_age,
        engine_cc=engine_cc,
    )

    pred = rf_model.predict(input_df)[0]
    formatted_price = f"Rp {pred:,.0f}".replace(",", ".")

    print(f"Estimasi Harga Mobil: {formatted_price}")

    price_card = f"""
## 🚗 Ringkasan Kendaraan

| Informasi  | Nilai             |
|------------|-------------------|
| Brand      | {brand}           |
| Model      | {car_model}       |
| Tahun      | {year}            |
| Umur Mobil | {car_age} tahun   |
| Engine CC  | {engine_cc} cc    |
| Transmisi  | {transmission}    |
| Lokasi     | {location}        |
| Mileage    | {mileage:,.0f} km |

---

# 💰 Prediksi Harga

## {formatted_price}

🟢 Estimasi harga berdasarkan model
"""

    ai_insight = generate_ai_insight(
        brand, car_model, year, location,
        mileage, transmission, engine_cc, formatted_price
    )

    return price_card, ai_insight

print("✅ Fungsi predict_car_price berhasil didefinisikan.")

✅ Fungsi predict_car_price berhasil didefinisikan.


In [ ]:
# ================================================================
# CELL 10 — Bangun & Jalankan Aplikasi Gradio
#            Tema: Dark Navy + Orange Accent (profesional)
# ================================================================

theme = gr.themes.Base(
    primary_hue=gr.themes.colors.orange,
    secondary_hue=gr.themes.colors.slate,
    neutral_hue=gr.themes.colors.slate,
    font=[
        gr.themes.GoogleFont("Inter"),
        gr.themes.GoogleFont("DM Sans"),
        "ui-sans-serif",
        "sans-serif"
    ],
    font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "monospace"],
).set(
    # Background utama
    body_background_fill="#0f1117",
    body_background_fill_dark="#0f1117",

    # Background block / card
    block_background_fill="#1a1d27",
    block_background_fill_dark="#1a1d27",

    # Border
    block_border_color="#2a2d3e",
    block_border_color_dark="#2a2d3e",
    block_border_width="1px",

    # Label
    block_label_background_fill="#252836",
    block_label_background_fill_dark="#252836",
    block_label_text_color="#e2e8f0",
    block_label_text_color_dark="#e2e8f0",

    # Body text
    body_text_color="#e2e8f0",
    body_text_color_dark="#e2e8f0",

    # Input / Textarea
    input_background_fill="#252836",
    input_background_fill_dark="#252836",
    input_border_color="#2a2d3e",
    input_border_color_dark="#2a2d3e",
    input_border_color_focus="#f97316",
    input_border_color_focus_dark="#f97316",
    input_placeholder_color="#64748b",
    input_placeholder_color_dark="#64748b",

    # Button primary (orange)
    button_primary_background_fill="#f97316",
    button_primary_background_fill_dark="#f97316",
    button_primary_background_fill_hover="#ea6c0a",
    button_primary_background_fill_hover_dark="#ea6c0a",
    button_primary_text_color="#ffffff",
    button_primary_text_color_dark="#ffffff",

    # Button secondary (dark gray)
    button_secondary_background_fill="#252836",
    button_secondary_background_fill_dark="#252836",
    button_secondary_background_fill_hover="#2e3247",
    button_secondary_background_fill_hover_dark="#2e3247",
    button_secondary_text_color="#e2e8f0",
    button_secondary_text_color_dark="#e2e8f0",

    # Slider & Checkbox accent
    checkbox_background_color="#252836",
    checkbox_background_color_dark="#252836",
    checkbox_background_color_selected="#f97316",
    checkbox_background_color_selected_dark="#f97316",
    checkbox_border_color="#2a2d3e",
    checkbox_border_color_dark="#2a2d3e",

    # Shadow
    block_shadow="0 4px 24px 0 rgba(0,0,0,0.4)",
    block_shadow_dark="0 4px 24px 0 rgba(0,0,0,0.4)",
)

css = """
/* ── Google Fonts ── */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=DM+Sans:wght@400;500;600&family=JetBrains+Mono&display=swap');

/* ── Global ── */
* { box-sizing: border-box; }

.gradio-container {
    max-width: 1300px !important;
    margin: auto;
    font-family: 'Inter', 'DM Sans', sans-serif !important;
    background-color: #0f1117 !important;
}

/* ── Header Markdown ── */
h1, h2, h3 {
    font-family: 'Inter', sans-serif !important;
    font-weight: 700 !important;
    color: #f1f5f9 !important;
    letter-spacing: -0.3px;
}

h1 { font-size: 2rem !important; }
h2 { font-size: 1.25rem !important; color: #f97316 !important; }

/* ── Section Label ── */
.gradio-container label span {
    font-family: 'Inter', sans-serif !important;
    font-size: 0.85rem !important;
    font-weight: 600 !important;
    color: #94a3b8 !important;
    text-transform: uppercase;
    letter-spacing: 0.8px;
}

/* ── Input / Dropdown / Number ── */
input, select, textarea {
    font-family: 'Inter', sans-serif !important;
    font-size: 0.95rem !important;
    color: #e2e8f0 !important;
    background-color: #252836 !important;
    border: 1px solid #2a2d3e !important;
    border-radius: 10px !important;
    transition: border-color 0.2s ease;
}

input:focus, select:focus, textarea:focus {
    border-color: #f97316 !important;
    outline: none !important;
    box-shadow: 0 0 0 3px rgba(249,115,22,0.15) !important;
}

/* ── Block / Card ── */
.block {
    background-color: transparent !important;
    border: none !important;
    border-radius: 0 !important;
    box-shadow: none !important;
    padding: 8px !important;
}
/* ── Primary Button (orange) ── */
button.primary {
    font-family: 'Inter', sans-serif !important;
    font-weight: 700 !important;
    font-size: 1rem !important;
    background: linear-gradient(135deg, #f97316, #ea580c) !important;
    border: none !important;
    border-radius: 12px !important;
    color: #fff !important;
    letter-spacing: 0.3px;
    box-shadow: 0 4px 16px rgba(249,115,22,0.35) !important;
    transition: all 0.2s ease !important;
}

button.primary:hover {
    background: linear-gradient(135deg, #fb923c, #f97316) !important;
    box-shadow: 0 6px 24px rgba(249,115,22,0.5) !important;
    transform: translateY(-1px);
}

/* ── Secondary Button ── */
button.secondary {
    font-family: 'Inter', sans-serif !important;
    font-weight: 600 !important;
    background-color: #252836 !important;
    border: 1px solid #2a2d3e !important;
    border-radius: 12px !important;
    color: #94a3b8 !important;
    transition: all 0.2s ease !important;
}

button.secondary:hover {
    background-color: #2e3247 !important;
    color: #e2e8f0 !important;
}

/* ── Radio Button ── */
.wrap label {
    color: #cbd5e1 !important;
    font-size: 0.9rem !important;
    font-weight: 500 !important;
}

/* ── Markdown Output ── */
.prose, .markdown {
    font-family: 'Inter', sans-serif !important;
    color: #e2e8f0 !important;
    line-height: 1.75 !important;
}

.prose table {
    border-collapse: collapse !important;
    width: 100% !important;
    border-radius: 10px !important;
    overflow: hidden !important;
}

.prose th {
    background-color: #f97316 !important;
    color: #fff !important;
    font-weight: 700 !important;
    padding: 10px 14px !important;
    text-align: left !important;
}

.prose td {
    background-color: #1e2130 !important;
    color: #e2e8f0 !important;
    padding: 9px 14px !important;
    border-bottom: 1px solid #2a2d3e !important;
}

.prose tr:hover td {
    background-color: #252836 !important;
}

.prose h1, .prose h2, .prose h3 {
    color: #f1f5f9 !important;
    font-weight: 700 !important;
}

/* ── Divider ── */
hr {
    border: none !important;
    border-top: 1px solid #2a2d3e !important;
    margin: 20px 0 !important;
}

/* ── Footer ── */
footer { display: none !important; }
.custom-footer {
    text-align: center;
    padding: 16px 0;
    color: #475569;
    font-size: 0.82rem;
    font-family: 'Inter', sans-serif;
    letter-spacing: 0.4px;
}
"""

with gr.Blocks(title="Prediksi Harga Mobil Bekas", theme=theme, css=css) as demo:

    gr.Info(
        "🚀 Langkah: Pilih Brand → Model → Tahun → Lokasi & Plat → Klik Prediksi"
    )

    # ── Header ──
    gr.Markdown("""
<div align="center">

# 🚗 AI Used Car Price Prediction

### Prediksi Harga Mobil Bekas | Model Machine Learning + Groq AI

<p style="font-size:15px; color:#94a3b8; font-family:'Inter',sans-serif;">
Masukkan spesifikasi kendaraan untuk mendapatkan estimasi harga beserta analisis AI.
</p>

</div>

---
""")

    # ── Baris 1: Brand – Model – Tahun ──
    gr.Markdown("## 🚘 Informasi Kendaraan")
    with gr.Row():
        brand     = gr.Dropdown(choices=brands, label="Brand")
        car_model = gr.Dropdown(choices=[], label="Model")
        year      = gr.Dropdown(choices=[], label="Tahun")


    # ── Baris 2: Lokasi ──
    gr.Markdown("## 📍 Lokasi")
    with gr.Row():
        location = gr.Dropdown(choices=locations, label="Lokasi Penjual")

    # ── Baris 3: Detail Teknis ──
    gr.Markdown("## 📋 Detail Teknis")
    with gr.Row():
        mileage      = gr.Number(label="Mileage (km)", value=50000)
        transmission = gr.Radio(
            choices=["Automatic", "Manual"],
            value="Automatic",
            label="Transmisi"
        )
        engine_cc = gr.Number(label="Engine CC", value=1500, interactive=True)

    # ── Tombol ──
    predict_btn = gr.Button(
        "🚀  Prediksi Harga Sekarang",
        variant="primary",
        size="lg"
    )

    # ── Output ──
    gr.Markdown("---")
    gr.Markdown("## 📈 Hasil Prediksi")
    with gr.Row():
        predicted_price = gr.Markdown()
        ai_result       = gr.Markdown()

    # ── Event Bindings ──
    brand.change(fn=update_models, inputs=brand, outputs=car_model)
    car_model.change(fn=update_years, inputs=[brand, car_model], outputs=year)
    year.change(fn=auto_fill_engine_cc, inputs=[car_model, year], outputs=engine_cc)

    predict_btn.click(
    fn=predict_car_price,
    inputs=[brand, car_model, year, location, mileage, transmission, engine_cc],
    outputs=[predicted_price, ai_result]
    )

    # ── Footer ──
    gr.Markdown("""
<div class="custom-footer">
Developed with Alka Tadra Koswara &nbsp;|&nbsp; Skripsi &nbsp;|&nbsp; • Gradio • Groq AI
</div>
""")

demo.launch(debug=False, share=True)

/tmp/ipykernel_3937/19492744.py:240: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Prediksi Harga Mobil Bekas", theme=theme, css=css) as demo:


🚀 Langkah: Pilih Brand → Model → Tahun → Lokasi & Plat → Klik Prediksi
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eba51610d827329ac1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
